# Results Plots Script

## Initializations

### Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Constants

In [ ]:
BASE_PATH = "./results"
DATASETS = ["adult", "london_house_price"]
ALGOS = ["oka", "kmember", "classic_mondrian"]
K = [2] + list(range(5, 101, 5))

In [ ]:
DATASET_CODES = {"adult": "adult", "london_house_price": "housing"}

## Plots

### Data Utility

In [ ]:
def get_ut_score(results_df):
    return results_df.set_index("metric").T.to_dict(orient="records")[0]


def get_all_ut_scores(_path, all_k):
    all_ut_scores = []
    for k in all_k:
        all_ut_scores.append(get_ut_score(pd.read_csv(f"{_path}/k_{k}/results_ut.csv")))
    return all_ut_scores

In [ ]:
def get_attack_result(path):
    return pd.read_csv(path)["correct"].sum().item()

In [ ]:
def plot(ALGO):
    all_algos_ut_scores = {}
    for _dataset in DATASETS:
        all_algos_ut_scores[f"{ALGO} ({DATASET_CODES[_dataset]})"] = get_all_ut_scores(
            f"{BASE_PATH}/{_dataset}/{ALGO}", K
        )
    all_attack_results = {}
    for _dataset in DATASETS:
        all_attack_results[f"{ALGO} ({DATASET_CODES[_dataset]})"] = [
            pd.read_csv(f"{BASE_PATH}/{_dataset}/{ALGO}/k_{k}/results_attack.csv")[
                "correct"
            ]
            .sum()
            .item()
            for k in K
        ]

    fig, ax = plt.subplots(1, 2, figsize=[8, 3])

    axis = ax[0]
    for algo in list(all_algos_ut_scores):
        axis.plot(
            K,
            [x["IL"] for x in all_algos_ut_scores[algo]],
            label=algo,
            # color=COLORS[algo],
            marker=("x" if algo.endswith("(adult)") else "p"),
            linestyle=":",
        )
    
    axis.legend(fontsize="medium")
    axis.set_ylim([-0.02, 0.72])
    axis.set_ylabel("NCP")
    axis.set_xlabel("k")
    axis.set_xlim([0, 105])
    axis.set_xticks(range(10, 101, 10))
    axis.set_yticks(np.arange(0, 0.75, 0.1))
    axis.grid(linestyle=":")
    
    axis = ax[1]
    
    for algo in list(all_attack_results):
        axis.plot(
            K,
            np.array(all_attack_results[algo]) / 3000,
            label=algo,
            # color=COLORS[algo],
            marker=("x" if algo.endswith("(adult)") else "p"),
            linestyle=":",
        )
    
    axis.legend(fontsize="medium")
    axis.set_ylim([0, 0.55])
    axis.set_ylabel("Vulnerability (Match Ratio)")
    axis.set_xlabel("k")
    axis.set_xlim([0, 105])
    axis.set_xticks(range(10, 101, 10))
    axis.set_yticks(np.arange(0, 0.56, 0.05))
    axis.grid(linestyle=":")
    
    fig.tight_layout(h_pad=0.05, w_pad=1.5)
    
    fig.savefig(
        f"./figs/result_{ALGO}_compared_NCP_vulnerability.png",
        dpi=150,
        bbox_inches="tight",
        pad_inches=0.015,
    )

In [ ]:
plot("kmember")

In [ ]:
plot("oka")

In [ ]:
plot("classic_mondrian")